In [1]:
%pip -q install duckdb pyarrow aiohttp

from google.colab import drive, userdata
from pathlib import Path
import asyncio
import json
import os
import random
import time

import aiohttp
import duckdb
import pandas as pd

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

PARQUET_PATH = Path(
    "/content/drive/MyDrive/Language Detection/"
    "sessions_lang_transcript_2026-08-23_2026-08-24.parquet"
)

if not PARQUET_PATH.is_file():
    raise FileNotFoundError(PARQUET_PATH)

OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise RuntimeError("OPENROUTER_API_KEY is missing from Colab Secrets")

OPENROUTER_CHAT_URL = "https://openrouter.ai/api/v1/chat/completions"
OPENROUTER_GENERATION_URL = "https://openrouter.ai/api/v1/generation"

MODELS = {
    # yesterday
    "gpt_oss_120b": "openai/gpt-oss-120b",
    "hermes_4_70b": "nousresearch/hermes-4-70b",
    "llama_3_3_70b": "meta-llama/llama-3.3-70b-instruct",
    # today
    "qwen3_8_27b": "qwen/qwen3.8-27b",
    "gemini_3_7_flash": "google/gemini-3.7-flash",
    "gpt_5_6_luna": "openai/gpt-5.6-luna",
    "nemotron_3_5_lightning": "nvidia/nemotron-3.5-lightning",
    "ox_alpha": "stealth/ox-alpha",
    "deepseek_v4_flash_0731": "deepseek/deepseek-v4-flash-0731",
}

LANGUAGE_NAMES = {
    "en": "English",
    "de": "German",
    "fr": "French",
    "pt": "Portuguese",
    "es": "Spanish",
    "ru": "Russian",
}

ALLOWED_CODES = tuple(LANGUAGE_NAMES)
SEGMENTS_PER_LANGUAGE = 25
REQUEST_TIMEOUT_SECONDS = 120
MAX_RETRIES = 4

HEADERS = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json",
}

con = duckdb.connect()
con.execute("SET threads TO 4")
con.execute("SET preserve_insertion_order = false")

def load_language_segments(language_code):
    if language_code not in LANGUAGE_NAMES:
        raise ValueError(language_code)

    frame = con.execute(
        """
        WITH sessions AS (
            SELECT
                gamesession_id,
                url,
                TRY_CAST(created_at AS TIMESTAMP) AS created_at,
                transcript_segments
            FROM read_parquet(?)
            WHERE
                lang_detected = ?
                AND transcript_segments IS NOT NULL
                AND len(transcript_segments) > 0
        ),
        exploded AS (
            SELECT
                gamesession_id,
                url,
                created_at,
                generate_subscripts(transcript_segments, 1) AS segment_index,
                UNNEST(transcript_segments) AS segment
            FROM sessions
        ),
        eligible AS (
            SELECT
                gamesession_id,
                url,
                created_at,
                segment_index,
                TRIM(segment.text) AS segment_text,
                len(
                    regexp_extract_all(
                        TRIM(segment.text),
                        '[\\p{L}\\p{N}]+'
                    )
                ) AS word_count
            FROM exploded
            WHERE
                segment.text IS NOT NULL
                AND TRIM(segment.text) <> ''
        ),
        ranked_sessions AS (
            SELECT
                gamesession_id,
                MAX(created_at) AS created_at,
                COUNT(*) AS eligible_segments
            FROM eligible
            WHERE word_count >= 4
            GROUP BY gamesession_id
            HAVING COUNT(*) >= ?
            ORDER BY created_at DESC NULLS LAST, gamesession_id DESC
            LIMIT 1
        )
        SELECT
            ? AS dataset_language,
            e.gamesession_id,
            e.url,
            e.segment_index,
            e.segment_text
        FROM eligible e
        INNER JOIN ranked_sessions s USING (gamesession_id)
        WHERE e.word_count >= 4
        ORDER BY e.segment_index
        LIMIT ?
        """,
        [
            PARQUET_PATH.as_posix(),
            language_code,
            SEGMENTS_PER_LANGUAGE,
            language_code,
            SEGMENTS_PER_LANGUAGE,
        ],
    ).df()

    if len(frame) != SEGMENTS_PER_LANGUAGE:
        raise RuntimeError(
            f"{language_code}: expected {SEGMENTS_PER_LANGUAGE} segments, found {len(frame)}"
        )

    frame.insert(0, "benchmark_row_id", range(len(frame)))
    return frame

def normalize_prediction(value):
    if not isinstance(value, str):
        return None

    value = value.strip().lower()
    aliases = {
        "english": "en",
        "german": "de",
        "deutsch": "de",
        "french": "fr",
        "français": "fr",
        "francais": "fr",
        "portuguese": "pt",
        "português": "pt",
        "portugues": "pt",
        "spanish": "es",
        "español": "es",
        "espanol": "es",
        "russian": "ru",
        "русский": "ru",
    }

    if value in ALLOWED_CODES:
        return value

    if value in aliases:
        return aliases[value]

    cleaned = value.replace("`", "").replace('"', "").replace("'", "").strip()

    if cleaned in ALLOWED_CODES:
        return cleaned

    if cleaned in aliases:
        return aliases[cleaned]

    for code in ALLOWED_CODES:
        if cleaned.startswith(code + " ") or cleaned.startswith(code + "\n"):
            return code

    return None

def build_payload(model_id, text):
    payload = {
        "model": model_id,
        "messages": [
            {
                "role": "system",
                "content": (
                    "Detect the language of the transcript. "
                    "Return exactly one ISO 639-1 code. "
                ),
            },
            {
                "role": "user",
                "content": text,
            },
        ],
        "temperature": 0,
    }

    if model_id == "openai/gpt-oss-120b":
        payload["reasoning"] = {"effort": "low"}

    return payload

async def post_chat(session, model_name, model_id, row):
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        started = time.perf_counter()

        try:
            async with session.post(
                OPENROUTER_CHAT_URL,
                headers=HEADERS,
                json=build_payload(model_id, row.segment_text),
            ) as response:
                elapsed = time.perf_counter() - started
                data = await response.json(content_type=None)

                if response.status == 200:
                    message = ((data.get("choices") or [{}])[0].get("message") or {})
                    content = message.get("content")
                    usage = data.get("usage") or {}

                    return {
                        "benchmark_row_id": row.benchmark_row_id,
                        "model_name": model_name,
                        "model_id": model_id,
                        "prediction": normalize_prediction(content),
                        "raw_prediction": content,
                        "generation_id": data.get("id"),
                        "http_status": response.status,
                        "request_duration_seconds": elapsed,
                        "prompt_tokens": usage.get("prompt_tokens"),
                        "completion_tokens": usage.get("completion_tokens"),
                        "total_tokens": usage.get("total_tokens"),
                        "response_cost_usd": usage.get("cost"),
                        "error": None,
                    }

                error = data.get("error")
                last_error = error.get("message") if isinstance(error, dict) else str(error or data)

                if response.status not in {408, 409, 429, 500, 502, 503, 504}:
                    break

                retry_after = response.headers.get("Retry-After")
                delay = float(retry_after) if retry_after else min(30.0, 2 ** attempt + random.random())
                await asyncio.sleep(delay)

        except Exception as exc:
            elapsed = time.perf_counter() - started
            last_error = f"{type(exc).__name__}: {exc}"
            if attempt < MAX_RETRIES:
                await asyncio.sleep(min(30.0, 2 ** attempt + random.random()))

    return {
        "benchmark_row_id": row.benchmark_row_id,
        "model_name": model_name,
        "model_id": model_id,
        "prediction": None,
        "raw_prediction": None,
        "generation_id": None,
        "http_status": None,
        "request_duration_seconds": elapsed,
        "prompt_tokens": None,
        "completion_tokens": None,
        "total_tokens": None,
        "response_cost_usd": None,
        "error": last_error,
    }

async def benchmark_language(language_code):
    segments = load_language_segments(language_code)
    timeout = aiohttp.ClientTimeout(total=REQUEST_TIMEOUT_SECONDS)
    records = []

    async with aiohttp.ClientSession(timeout=timeout) as session:
        for row in segments.itertuples(index=False):
            for model_name, model_id in MODELS.items():
                record = await post_chat(session, model_name, model_id, row)
                records.append(record)

    raw = pd.DataFrame(records)

    predictions = (
        raw.pivot(
            index="benchmark_row_id",
            columns="model_name",
            values="prediction",
        )
        .rename(columns=lambda name: f"openrouter_{name}_language")
        .reset_index()
    )

    result = segments.merge(predictions, on="benchmark_row_id", how="left")
    return result, raw

async def fetch_generation(session, generation_id):
    started = time.perf_counter()

    try:
        async with session.get(
            OPENROUTER_GENERATION_URL,
            headers=HEADERS,
            params={"id": generation_id},
        ) as response:
            lookup_duration = time.perf_counter() - started
            payload = await response.json(content_type=None)

            if response.status != 200:
                return {
                    "generation_id": generation_id,
                    "generation_lookup_http_status": response.status,
                    "generation_lookup_seconds": lookup_duration,
                    "generation_error": str(payload),
                }

            data = payload.get("data") or payload
            return {
                "generation_id": generation_id,
                "generation_lookup_http_status": response.status,
                "generation_lookup_seconds": lookup_duration,
                "generation_model": data.get("model"),
                "generation_provider": data.get("provider_name") or data.get("provider"),
                "generation_prompt_tokens": data.get("tokens_prompt"),
                "generation_completion_tokens": data.get("tokens_completion"),
                "generation_total_cost_usd": data.get("total_cost"),
                "generation_latency_ms": data.get("latency"),
                "generation_generation_time_ms": data.get("generation_time"),
                "generation_error": None,
            }

    except Exception as exc:
        return {
            "generation_id": generation_id,
            "generation_lookup_http_status": None,
            "generation_lookup_seconds": time.perf_counter() - started,
            "generation_error": f"{type(exc).__name__}: {exc}",
        }

async def build_accounting(raw_results):
    source = raw_results[
        raw_results["generation_id"].notna()
    ].copy()

    timeout = aiohttp.ClientTimeout(total=REQUEST_TIMEOUT_SECONDS)
    metadata = []

    async with aiohttp.ClientSession(timeout=timeout) as session:
        for generation_id in source["generation_id"].drop_duplicates():
            metadata.append(await fetch_generation(session, generation_id))

    metadata = pd.DataFrame(metadata)

    if metadata.empty:
        detailed = source.copy()
    else:
        detailed = source.merge(metadata, on="generation_id", how="left")

    summary = (
        detailed.groupby(["model_name", "model_id"], as_index=False)
        .agg(
            api_calls=("generation_id", "size"),
            successful_predictions=("prediction", lambda x: x.notna().sum()),
            prompt_tokens=("prompt_tokens", "sum"),
            completion_tokens=("completion_tokens", "sum"),
            total_tokens=("total_tokens", "sum"),
            response_cost_usd=("response_cost_usd", "sum"),
            request_duration_seconds=("request_duration_seconds", "sum"),
            avg_request_duration_seconds=("request_duration_seconds", "mean"),
            official_generation_cost_usd=("generation_total_cost_usd", "sum"),
            avg_official_latency_ms=("generation_latency_ms", "mean"),
            avg_official_generation_time_ms=("generation_generation_time_ms", "mean"),
        )
    )

    return detailed, summary

print("Ready")
print("Parquet:", PARQUET_PATH)
print("Models :", ", ".join(MODELS))
print("Flow   : run language cells one by one, then accounting cells one by one")


Ready
Parquet: /content/drive/MyDrive/Language Detection/sessions_lang_transcript_2026-08-23_2026-08-24.parquet
Models : gpt_oss_120b, hermes_4_70b, llama_3_3_70b, qwen3_8_27b, gemini_3_7_flash, gpt_5_6_luna, nemotron_3_5_lightning, ox_alpha, deepseek_v4_flash_0731
Flow   : run language cells one by one, then accounting cells one by one


In [2]:
english_benchmark, english_raw = await benchmark_language("en")

print("English")
print("Segments:", len(english_benchmark))
print("API calls:", len(english_raw))
print("Successful predictions:", int(english_raw["prediction"].notna().sum()))
print("Failed predictions:", int(english_raw["prediction"].isna().sum()))

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(english_benchmark)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

English
Segments: 25
API calls: 225
Successful predictions: 208
Failed predictions: 17


,benchmark_row_id,dataset_language,gamesession_id,url,segment_index,segment_text,openrouter_deepseek_v4_flash_0731_language,openrouter_gemini_3_7_flash_language,openrouter_gpt_5_6_luna_language,openrouter_gpt_oss_120b_language,openrouter_hermes_4_70b_language,openrouter_llama_3_3_70b_language,openrouter_nemotron_3_5_lightning_language,openrouter_ox_alpha_language,openrouter_qwen3_8_27b_language
0,0,en,141268501,https://www.twitch.tv/videos/2854311727,2,"Don't be talking, shut my door.",None,en,en,en,en,en,en,en,en
1,1,en,141268501,https://www.twitch.tv/videos/2854311727,3,We'll start it.,en,en,en,en,en,en,en,en,en
2,2,en,141268501,https://www.twitch.tv/videos/2854311727,4,We'll start with the basics.,en,en,en,en,en,en,en,en,en
3,3,en,141268501,https://www.twitch.tv/videos/2854311727,5,actually showed up. You guys owe me ten bucks.,None,en,en,en,en,en,en,en,en
4,4,en,141268501,https://www.twitch.tv/videos/2854311727,6,"personal. It's just, uh, we're stuck in Groundhog Day out here.",en,en,en,en,en,en,en,en,en
5,5,en,141268501,https://www.twitch.tv/videos/2854311727,7,"trying to, you know, not go crazy..",en,en,en,en,en,en,en,en,en
6,6,en,141268501,https://www.twitch.tv/videos/2854311727,8,"Okay, it's simple really, shout next and the guys will let the survivor in.",en,en,en,en,en,None,en,en,en
7,7,en,141268501,https://www.twitch.tv/videos/2854311727,9,"Next survivor. First, do a simple inspection.",en,en,en,en,en,None,en,en,en
8,8,en,141268501,https://www.twitch.tv/videos/2854311727,10,Take the flashlight from the table.,en,en,en,en,en,None,en,en,en
9,9,en,141268501,https://www.twitch.tv/videos/2854311727,11,Take your time. Match what you see against the symptom chart.,None,en,en,en,en,None,None,en,en


In [ ]:
german_benchmark, german_raw = await benchmark_language("de")

print("German")
print("Segments:", len(german_benchmark))
print("API calls:", len(german_raw))
print("Successful predictions:", int(german_raw["prediction"].notna().sum()))
print("Failed predictions:", int(german_raw["prediction"].isna().sum()))

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(german_benchmark)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
french_benchmark, french_raw = await benchmark_language("fr")

print("French")
print("Segments:", len(french_benchmark))
print("API calls:", len(french_raw))
print("Successful predictions:", int(french_raw["prediction"].notna().sum()))
print("Failed predictions:", int(french_raw["prediction"].isna().sum()))

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(french_benchmark)


In [ ]:
portuguese_benchmark, portuguese_raw = await benchmark_language("pt")

print("Portuguese")
print("Segments:", len(portuguese_benchmark))
print("API calls:", len(portuguese_raw))
print("Successful predictions:", int(portuguese_raw["prediction"].notna().sum()))
print("Failed predictions:", int(portuguese_raw["prediction"].isna().sum()))

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(portuguese_benchmark)


In [ ]:
spanish_benchmark, spanish_raw = await benchmark_language("es")

print("Spanish")
print("Segments:", len(spanish_benchmark))
print("API calls:", len(spanish_raw))
print("Successful predictions:", int(spanish_raw["prediction"].notna().sum()))
print("Failed predictions:", int(spanish_raw["prediction"].isna().sum()))

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(spanish_benchmark)


In [ ]:
russian_benchmark, russian_raw = await benchmark_language("ru")

print("Russian")
print("Segments:", len(russian_benchmark))
print("API calls:", len(russian_raw))
print("Successful predictions:", int(russian_raw["prediction"].notna().sum()))
print("Failed predictions:", int(russian_raw["prediction"].isna().sum()))

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(russian_benchmark)


In [ ]:
english_accounting, english_accounting_summary = await build_accounting(english_raw)

print("English accounting")
display(english_accounting_summary)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(
        english_accounting[
            [
                "benchmark_row_id",
                "model_name",
                "model_id",
                "prediction",
                "generation_id",
                "prompt_tokens",
                "completion_tokens",
                "total_tokens",
                "response_cost_usd",
                "request_duration_seconds",
                "generation_provider",
                "generation_prompt_tokens",
                "generation_completion_tokens",
                "generation_total_cost_usd",
                "generation_latency_ms",
                "generation_generation_time_ms",
                "generation_error",
            ]
        ]
    )


In [ ]:
german_accounting, german_accounting_summary = await build_accounting(german_raw)

print("German accounting")
display(german_accounting_summary)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(
        german_accounting[
            [
                "benchmark_row_id",
                "model_name",
                "model_id",
                "prediction",
                "generation_id",
                "prompt_tokens",
                "completion_tokens",
                "total_tokens",
                "response_cost_usd",
                "request_duration_seconds",
                "generation_provider",
                "generation_prompt_tokens",
                "generation_completion_tokens",
                "generation_total_cost_usd",
                "generation_latency_ms",
                "generation_generation_time_ms",
                "generation_error",
            ]
        ]
    )


In [ ]:
french_accounting, french_accounting_summary = await build_accounting(french_raw)

print("French accounting")
display(french_accounting_summary)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(
        french_accounting[
            [
                "benchmark_row_id",
                "model_name",
                "model_id",
                "prediction",
                "generation_id",
                "prompt_tokens",
                "completion_tokens",
                "total_tokens",
                "response_cost_usd",
                "request_duration_seconds",
                "generation_provider",
                "generation_prompt_tokens",
                "generation_completion_tokens",
                "generation_total_cost_usd",
                "generation_latency_ms",
                "generation_generation_time_ms",
                "generation_error",
            ]
        ]
    )


In [ ]:
portuguese_accounting, portuguese_accounting_summary = await build_accounting(portuguese_raw)

print("Portuguese accounting")
display(portuguese_accounting_summary)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(
        portuguese_accounting[
            [
                "benchmark_row_id",
                "model_name",
                "model_id",
                "prediction",
                "generation_id",
                "prompt_tokens",
                "completion_tokens",
                "total_tokens",
                "response_cost_usd",
                "request_duration_seconds",
                "generation_provider",
                "generation_prompt_tokens",
                "generation_completion_tokens",
                "generation_total_cost_usd",
                "generation_latency_ms",
                "generation_generation_time_ms",
                "generation_error",
            ]
        ]
    )


In [ ]:
spanish_accounting, spanish_accounting_summary = await build_accounting(spanish_raw)

print("Spanish accounting")
display(spanish_accounting_summary)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(
        spanish_accounting[
            [
                "benchmark_row_id",
                "model_name",
                "model_id",
                "prediction",
                "generation_id",
                "prompt_tokens",
                "completion_tokens",
                "total_tokens",
                "response_cost_usd",
                "request_duration_seconds",
                "generation_provider",
                "generation_prompt_tokens",
                "generation_completion_tokens",
                "generation_total_cost_usd",
                "generation_latency_ms",
                "generation_generation_time_ms",
                "generation_error",
            ]
        ]
    )


In [ ]:
russian_accounting, russian_accounting_summary = await build_accounting(russian_raw)

print("Russian accounting")
display(russian_accounting_summary)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(
        russian_accounting[
            [
                "benchmark_row_id",
                "model_name",
                "model_id",
                "prediction",
                "generation_id",
                "prompt_tokens",
                "completion_tokens",
                "total_tokens",
                "response_cost_usd",
                "request_duration_seconds",
                "generation_provider",
                "generation_prompt_tokens",
                "generation_completion_tokens",
                "generation_total_cost_usd",
                "generation_latency_ms",
                "generation_generation_time_ms",
                "generation_error",
            ]
        ]
    )
